# Introduction to NLP with Hugging Face Transformers 🤗

This notebook provides a comprehensive hands-on guide to using the `transformers` library for various Natural Language Processing (NLP) tasks. We cover everything from low-level model mechanics to high-level application pipelines.

### Key Concepts Covered:
1.  **Model Mechanics**: Tokenization, Logits, and Decoding Strategies.
2.  **Dataset Integration**: Loading and using the `datasets` library with Pandas.
3.  **Core NLP Tasks**:
    *   **Sentiment Analysis**: General and Specialized (Financial).
    *   **Named Entity Recognition (NER)**: Identifying entities in text.
    *   **Question Answering (QA)**: Extracting answers from context.
    *   **Machine Translation**: Converting text between languages.

---

## Environment Setup
Before starting, ensure you have the necessary environment and dependencies:
- **Environment**: `conda activate hugvenv312`
- **Installation**: `pip install -r requirements.txt`

### Useful Resources
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/): An interactive visualization of how Transformers work.
- [Hugging Face Documentation](https://huggingface.co/docs/transformers/index): Official documentation for the `transformers` library.

In [1]:
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore")

print("Jupyter Notebook Initialized: Hugging Face Transformers Exploration")

Jupyter Notebook Initialized: Hugging Face Transformers Exploration


In [2]:
# System Diagnostics: Check CPU, GPU, and Library Versions
import platform
import torch
import torch.nn.functional as F
import subprocess

print(f"--- System Information ---")
print(f"Platform: {platform.platform()}")
print(f"Python:   {platform.python_version()}")
print(f"PyTorch:  {torch.__version__}")

print(f"\n--- GPU/ROCm Accelerators ---")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Primary Device: {torch.cuda.get_device_name(0)}")
else:
    print("Optimization Note: No CUDA-capable GPU detected. Operations will run on CPU.")

--- System Information ---
Platform: Windows-11-10.0.26200-SP0
Python:   3.12.13
PyTorch:  2.9.1+rocm7.2.1

--- GPU/ROCm Accelerators ---
CUDA Available: True
Device Count:   1
Primary Device: AMD Radeon RX 7900 XT


## Method 1: Using the High-Level `pipeline` API

The `pipeline` is the easiest way to use a model for a specific task. It abstracts away the complexity of tokenization, model inference, and decoding.

We'll use GPT-2 for text generation.

In [3]:
from transformers import pipeline

# Initialize the text-generation pipeline with GPT-2
# This automatically downloads the model and tokenizer if not already present
generator = pipeline('text-generation', model='openai-community/gpt2')

# Define a prompt
prompt = 'What is machine learning?'

# Generate text
# The pipeline returns a list of dictionaries containing the generated text
results = generator(prompt, max_length=50, num_return_sequences=1)

# Print the result
print(results[0]['generated_text'])

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is machine learning? It's a technology that's been around for a long time (the notion of "learning" came up more than a generation ago on the Apple iPhone). Computers can learn and write code, but they're also able to


## Method 2: Manual Control with Models and Tokenizers

For more granular control, you can load the tokenizer and model independently. This allows you to inspect intermediate steps like token IDs and attention masks.

### Loading Model & Tokenizer
We use `AutoTokenizer` and `AutoModelForCausalLM` which automatically select the correct classes based on the model checkpoint.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'openai-community/gpt2'

# Load the tokenizer responsible for converting text to numbers
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load the causal language model
model = AutoModelForCausalLM.from_pretrained(model_id)

print(f"Tokenizer loaded: {type(tokenizer).__name__}")
print(f"Model loaded:     {type(model).__name__}")

Tokenizer loaded: GPT2TokenizerFast
Model loaded:     GPT2LMHeadModel


In [5]:
# Convert text to token IDs
sentence = 'What is machine learning?'
inputs = tokenizer(sentence, return_tensors='pt')
input_ids = inputs["input_ids"]

print(f"Input IDs: {input_ids}")

# Convert back to text
decoded_text = tokenizer.decode(input_ids[0])
print(f"Decoded:   {decoded_text}")

Input IDs: tensor([[2061,  318, 4572, 4673,   30]])
Decoded:   What is machine learning?


### Understanding Subword Tokenization

Modern tokenizers split words into "subwords" or "tokens". This allows the model to handle a large vocabulary with a smaller set of base units. 

Let's look at how the word `unbelievable` is decomposed.

In [6]:
word = 'unbelievable'
input_ids = tokenizer(word, return_tensors='pt').input_ids

# Show the IDs and the reconstructed word
print(f"Token IDs: {input_ids}")
print(f"Decoded:   {tokenizer.decode(input_ids[0])}")

Token IDs: tensor([[  403,  6667, 11203,   540]])
Decoded:   unbelievable


In [7]:
# Inspect individual tokens
for token_id in input_ids[0]:
    print(f"ID {token_id:5} -> '{tokenizer.decode(token_id)}'")

ID   403 -> 'un'
ID  6667 -> 'bel'
ID 11203 -> 'iev'
ID   540 -> 'able'


### Handling Rare Words (Out-of-Vocabulary)

When a word is rare (like `homoscendasticity`), the tokenizer breaks it down into even smaller phonetic or character-level fragments. This ensures every possible string can be represented.

In [8]:
complex_word = 'homoscendasticity'
ids = tokenizer(complex_word, return_tensors='pt').input_ids

print(f"Breakdown for '{complex_word}':")
for token_id in ids[0]:
    print(f"ID {token_id:5} -> '{tokenizer.decode(token_id)}'")

full_decoded = tokenizer.decode(ids.squeeze())
print(f"\nReassembled: {full_decoded}")

Breakdown for 'homoscendasticity':
ID 26452 -> 'hom'
ID   418 -> 'os'
ID 15695 -> 'cend'
ID  3477 -> 'astic'
ID   414 -> 'ity'

Reassembled: homoscendasticity


## Probability Distributions and Logits

Language models predict the next token by outputting "logits" (raw scores) for every token in their vocabulary. These scores are then converted into probabilities.

Let's see what the model thinks should come next in a sentence.

In [9]:
# Get model outputs (logits) for the last token in the sequence
with torch.no_grad():
    outputs = model(input_ids)
    
last_token_logits = outputs.logits[0, -1]

# Find the most likely next token ID
predicted_id = last_token_logits.argmax()
predicted_word = tokenizer.decode(predicted_id)

print(f"Context:        '{tokenizer.decode(input_ids[0])}'")
print(f"Most likely:    '{predicted_word}'")

Context:        'unbelievable'
Most likely:    '"'


## Text Generation Strategies (Decoding)

There are several ways to choose the "next" token from the probability distribution. Each has trade-offs between accuracy and creativity:

1.  **Greedy Decoding**: Always pick the single most probable token. (Deterministic but can be repetitive).
2.  **Top-K Sampling**: Randomly sample from the top `k` probable tokens.
3.  **Top-P (Nucleus) Sampling**: Randomly sample from the smallest set of tokens whose cumulative probability $\ge p$.
4.  **Temperature Sampling**: Scaling the distribution before sampling. High temperature (>1) = more random; Low temperature (<1) = more confident.
5.  **Random Sampling**: Sample from the entire vocabulary based on the probabilities. (Highly unpredictable).

In [10]:
def greedy_decode(logits):
    """Selects the absolute most likely next token."""
    return torch.argmax(logits, dim=-1)

def top_k_sampling(logits, k=50):
    """Filters for the top-k most likely tokens, then samples from them."""
    values, indices = torch.topk(logits, k=k)
    probs = F.softmax(values, dim=-1)
    sampled_index = torch.multinomial(probs, num_samples=1)
    return indices[sampled_index]

def top_p_sampling(logits, p=0.9):
    """Samples from the smallest set of tokens that account for p% of the probability mass."""
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Mask tokens outside the nucleus
    mask = cumulative_probs > p
    # Shift mask to include the first token that exceeds p
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False
    
    sorted_logits[mask] = float('-inf')
    probs = F.softmax(sorted_logits, dim=-1)
    sampled_index = torch.multinomial(probs, num_samples=1)
    return sorted_indices[sampled_index]

def temperature_sampling(logits, temperature=1.0):
    """Distorts the distribution to increase or decrease randomness."""
    adjusted_logits = logits / max(temperature, 1e-5) # Avoid division by zero
    probs = F.softmax(adjusted_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

def random_sampling(logits):
    """Plain multinomial sampling from the full distribution."""
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [11]:
# Demonstrate different decoding predictions for a single prompt
prompt = 'Today I decided to go to the local'
inputs = tokenizer(prompt, return_tensors='pt').input_ids
with torch.no_grad():
    logits = model(inputs).logits[0, -1]

print(f"Prompt: '{prompt} ...'\n")
print(f"Greedy:      '{tokenizer.decode(greedy_decode(logits))}'")
print(f"Top-K:       '{tokenizer.decode(top_k_sampling(logits, k=5))}'")
print(f"Top-P:       '{tokenizer.decode(top_p_sampling(logits, p=0.8))}'")
print(f"Temp (1.5):  '{tokenizer.decode(temperature_sampling(logits, temperature=1.5))}'")
print(f"Random:      '{tokenizer.decode(random_sampling(logits))}'")

Prompt: 'Today I decided to go to the local ...'

Greedy:      ' library'
Top-K:       ' store'
Top-P:       ' auto'
Temp (1.5):  ' edition'
Random:      ' Funk'


### Comparing Decoding Strategies

Let's apply the different strategies we've defined to see how the generated output varies for the same prompt. This illustrates the trade-off between pickiness (Greedy) and creativity (Sampling).

In [12]:
# Visualizing the Probability Distribution
prompt = 'Machine learning is becoming very'
inputs = tokenizer(prompt, return_tensors='pt').input_ids

with torch.no_grad():
    logits = model(inputs).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)

# Get the top 10 most likely candidates
top_values, top_indices = torch.topk(probs, k=10)

print(f"Top 10 predictions for: '{prompt} ...'\n")
for val, idx in zip(top_values, top_indices):
    token_str = tokenizer.decode([idx])
    print(f"{token_str:15} | Probability: {val.item():.2%}")

Top 10 predictions for: 'Machine learning is becoming very ...'

 popular        | Probability: 28.36%
 powerful       | Probability: 6.24%
 important      | Probability: 5.42%
 complex        | Probability: 3.46%
 much           | Probability: 3.39%
 useful         | Probability: 2.63%
 sophisticated  | Probability: 2.27%
 difficult      | Probability: 2.17%
 common         | Probability: 1.80%
 interesting    | Probability: 1.42%


### Visualizing Predicted Token Probabilities

To better understand the model's "internal state," we can visualize the top candidates and their respective probabilities for the next word.

## Introduction to Datasets for Text Analysis

Hugging Face provides the `datasets` library to easily load and manage large-scale data for NLP tasks. Here, we'll look at the IMDB Movie Reviews dataset, commonly used for Sentiment Analysis.

In [13]:
from datasets import load_dataset

# Load the IMDB dataset (automatically caches locally)
dataset = load_dataset("stanfordnlp/imdb")

print(f"Dataset type: {type(dataset)}")
print(dataset)

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [14]:
import pandas as pd

# Convert the training split to a Pandas DataFrame for easier exploration
my_dataset_df = dataset['train'].to_pandas()

# Display the first few rows
print(f"Total rows in training set: {len(my_dataset_df)}")
my_dataset_df.head()

Total rows in training set: 25000


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [26]:
from transformers import pipeline

# Default model for sentiment analysis fine-tuned on the SST-2 dataset (model='distilbert-base-uncased-finetuned-sst-2-english')
# This model is optimized for sentiment classification and will return labels like 'POSITIVE' or 'NEGATIVE' along with confidence scores.
# Use GPU (device 0) if available for faster inference
classifier = pipeline('text-classification', 
                          model='distilbert-base-uncased-finetuned-sst-2-english', 
                          device= 0 if torch.cuda.is_available() else -1)

print("text : this day is great!")
print(classifier('this day is great!'))
print(" ")
print("text : this day is terrible and I am very sad about it.")
print(classifier('this day is terrible and I am very sad about it.'))

text : this day is great!
[{'label': 'POSITIVE', 'score': 0.999881386756897}]
 
text : this day is terrible and I am very sad about it.
[{'label': 'NEGATIVE', 'score': 0.9989134073257446}]


### Comparing Model Predictions with Labels

Now that we have a classifier, we can apply it to our dataset. We'll define a helper function to truncate text (as many models have a sequence length limit, typically 512 tokens) and apply the classifier to the IMDB reviews.

In [ ]:
def score(review_text):
    """Returns the sentiment score for a given review text."""
    # We truncate to 500 characters as requested
    return classifier(review_text[:500])[0]["label"]

# SPEED IMPROVEMENT: Use batch processing instead of .apply()
# Passing a list to the pipeline with a batch_size allows the GPU to work in parallel.
print(f"Processing {len(my_dataset_df)} reviews...")
texts_to_process = my_dataset_df["text"].str[:500].tolist()

# Using the pipeline's batching capability (significantly faster than .apply)
results = classifier(texts_to_process, batch_size=64)

# Map results back to labels
my_dataset_df["model_prediction"] = [res["label"] for res in results]
print("Processing complete.")

Re-initializing classifier on GPU for better performance...
Processing 25000 reviews...
Processing complete.


In [18]:
my_dataset_df

# To show only the relevant columns and a subset of rows, you can uncomment the following line:
# my_dataset_df[["label", "model_prediction"]][:20]

,text,label,model_prediction
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,NEGATIVE
1,"""I Am Curious: Yellow"" is a risible and preten...",0,NEGATIVE
2,If only to avoid making this type of film in t...,0,NEGATIVE
3,This film was probably inspired by Godard's Ma...,0,POSITIVE
4,"Oh, brother...after hearing about this ridicul...",0,NEGATIVE
...,...,...,...
24995,A hit at the time but now better categorised a...,1,NEGATIVE
24996,I love this movie like no other. Another time ...,1,POSITIVE
24997,This film and it's sequel Barry Mckenzie holds...,1,POSITIVE
24998,'The Adventures Of Barry McKenzie' started lif...,1,NEGATIVE


### Inspecting the Results

Let's look at the first few rows of our updated DataFrame to see how the model's sentiment prediction compares to the ground truth labels.

In [19]:
review = my_dataset_df.iloc[0]["text"]
print(classifier(review)[0]["label"])

POSITIVE


## Task 2: Specialized Sentiment Analysis (FinBERT)

While general models like `distilbert-base-uncased-finetuned-sst-2-english` are great for common text, specialized domains often require models trained on relevant data. 

**FinBERT** is a pre-trained NLP model to analyze the sentiment of financial text. It is built by further training the BERT language model in the financial domain, using a large financial corpus.

### Initializing the Financial Classifier
We use the `ProsusAI/finbert` model which categorizes text into **Positive**, **Negative**, or **Neutral** sentiments.

In [20]:
from transformers import pipeline

finbert_classifier = pipeline("sentiment-analysis", model="ProsusAI/finbert")

In [21]:
sentences = ["Strong consumer demand is driving growth in the retail sector across all regions.",
              "The company's stock price plummeted after the disappointing earnings report, causing concern among investors.",
              "The company reported a significant increase in revenue this quarter, exceeding market expectations."]

print(f'Sentences: {sentences}')
print(finbert_classifier(sentences))

Sentences: ['Strong consumer demand is driving growth in the retail sector across all regions.', "The company's stock price plummeted after the disappointing earnings report, causing concern among investors.", 'The company reported a significant increase in revenue this quarter, exceeding market expectations.']
[{'label': 'positive', 'score': 0.9189533591270447}, {'label': 'negative', 'score': 0.9662295579910278}, {'label': 'positive', 'score': 0.9554389715194702}]


## Task 3: Named Entity Recognition (NER)

Named Entity Recognition is the task of identifying and classifying key information (entities) in text into pre-defined categories such as:
- **PER**: Person
- **ORG**: Organization
- **LOC**: Location
- **MISC**: Miscellaneous

We'll use a standard NER pipeline which, by default, uses a model fine-tuned on the CoNLL-2003 dataset.

In [22]:
sentences_ner = ["Apple announced record earnings in the United States on monday.",
                 "I live in New York City and work for Google.",
                 "I am planning to visit the Eiffel Tower in Paris next summer."]

# Standard NER pipeline using a default model (e.g., 'dbmdz/bert-large-cased-finetuned-conll03-english')
ner_pipeline = pipeline("ner")

ner_pipeline(sentences_ner)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision f2482bf (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initiali

[[{'entity': 'I-ORG',
   'score': np.float32(0.99904615),
   'index': 1,
   'word': 'Apple',
   'start': 0,
   'end': 5},
  {'entity': 'I-LOC',
   'score': np.float32(0.99971646),
   'index': 7,
   'word': 'United',
   'start': 39,
   'end': 45},
  {'entity': 'I-LOC',
   'score': np.float32(0.9996525),
   'index': 8,
   'word': 'States',
   'start': 46,
   'end': 52}],
 [{'entity': 'I-LOC',
   'score': np.float32(0.99940705),
   'index': 4,
   'word': 'New',
   'start': 10,
   'end': 13},
  {'entity': 'I-LOC',
   'score': np.float32(0.99933845),
   'index': 5,
   'word': 'York',
   'start': 14,
   'end': 18},
  {'entity': 'I-LOC',
   'score': np.float32(0.9993162),
   'index': 6,
   'word': 'City',
   'start': 19,
   'end': 23},
  {'entity': 'I-ORG',
   'score': np.float32(0.9990883),
   'index': 10,
   'word': 'Google',
   'start': 37,
   'end': 43}],
 [{'entity': 'I-LOC',
   'score': np.float32(0.9373662),
   'index': 7,
   'word': 'E',
   'start': 27,
   'end': 28},
  {'entity': 'I-

## Task 4: Question Answering (QA)

The Question Answering task allows the model to extract an answer from a given context based on a provided question. This is known as **extractive question answering**.

### Initializing the QA Pipeline
The default pipeline uses a model like `distilbert-base-cased-distilled-squad`.

In [23]:
qa_bot = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 626af31 (https://huggingface.co/distilbert/distilbert-base-cased-distilled-squad).
Using a pipeline without specifying a model name and revision in production is not recommended.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [31]:
# Example: Extracting an answer from a context
context = """The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome that covers most of the Amazon basin of South America.
This basin encompasses 7 million square kilometers (2.7 million square miles), of which 5.5 million square kilometers (2.1 million square miles)
are covered by the rainforest. The Amazon represents over half of the planet's remaining rainforests and comprises the largest and most biodiverse\
tract of tropical rainforest in the world, with an estimated 390 billion individual trees divided into 16,000 species. The Amazon rainforest is home 
to an estimated 10% of the known species on Earth, including a vast array of plants, animals, and microorganisms. It plays a critical role in regulating 
the global climate by absorbing large amounts of carbon dioxide and producing oxygen. The rainforest also provides essential ecosystem services, such 
as water filtration, soil stabilization, and habitat for indigenous communities. However, it faces significant threats from deforestation, climate change, 
and human activities, which have led to concerns about its long-term sustainability and the loss of biodiversity."""

question = "Where is the Amazon rainforest located?"

result = qa_bot(question=question, context=context)

print(f"Context:  {context}\n")
print(f"Question: {question}\n")
print(f"Answer:   {result['answer']} (Score: {result['score']:.2f})")

Context:  The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome that covers most of the Amazon basin of South America.
This basin encompasses 7 million square kilometers (2.7 million square miles), of which 5.5 million square kilometers (2.1 million square miles)
are covered by the rainforest. The Amazon represents over half of the planet's remaining rainforests and comprises the largest and most biodiversetract of tropical rainforest in the world, with an estimated 390 billion individual trees divided into 16,000 species. The Amazon rainforest is home 
to an estimated 10% of the known species on Earth, including a vast array of plants, animals, and microorganisms. It plays a critical role in regulating 
the global climate by absorbing large amounts of carbon dioxide and producing oxygen. The rainforest also provides essential ecosystem services, such 
as water filtration, soil stabilization, and habitat for indigenous communities. However, it faces signifi

## Task 5: Machine Translation

Machine translation allows us to convert text from one language to another. Hugging Face pipelines support many language pairs using models like T5, MarianMT, and MBart.

### English-to-French Translation
By default, the `translation_en_to_fr` pipeline uses the **T5-base** model, which is a versatile "Text-to-Text Transfer Transformer."

In [ ]:
# Initialize translation pipeline
# Note: device=0 uses the GPU if available. 
# For T5, we use the specific 'translation_en_to_fr' task name.
translator = pipeline("translation_en_to_fr", device=0 if torch.cuda.is_available() else -1)

# List of example sentences to translate
sample_sentences = [
    "Hello, how are you?",
    "What is your name?",
    "The weather is nice today.",
    "I love learning new languages and exploring different cultures.",
    "Thank you for your help."
]

print("--- Translation Results (EN -> FR) ---\n")
for sent in sample_sentences:
    # The pipeline returns a list of dictionaries
    translation = translator(sent)[0]['translation_text']
    print(f"English: {sent}")
    print(f"French:  {translation}")
    print("-" * 30)

No model was supplied, defaulted to google-t5/t5-base and revision 686f1db (https://huggingface.co/google-t5/t5-base).
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'translation_text': 'Bonjour, comment êtes-vous?'}]

[{'translation_text': 'Quel est votre nom?'}]

[{'translation_text': "Le temps est agréable aujourd'hui."}]

[{'translation_text': "J'aime apprendre de nouvelles langues et explorer différentes cultures."}]

[{'translation_text': 'Merci de votre aide.'}]



## Conclusion and Next Steps

In this notebook, we've explored the versatility of the Hugging Face ecosystem. We've seen how `pipelines` make complex tasks accessible in just a few lines of code, while also diving deep into the tokenization and decoding strategies that power these models.

### Where to go from here?
- **Fine-Tuning**: Learn how to train these models on your own custom datasets.
- **Model Hub**: Explore the [Hugging Face Model Hub](https://huggingface.co/models) to find models for specialized domains (Medical, Legal, etc.).
- **Deployment**: Check out `Inference Endpoints` or `Text Generation Inference (TGI)` for production use cases.

Happy coding! 🚀